<a href="https://colab.research.google.com/github/Dacon-contest/dacon-data-segmentation_and_train/blob/main/5_fold_v5(%EB%A7%88%EC%8A%A4%ED%81%AC_%EB%B9%BC%EA%B8%B0).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import cv2
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler, autocast
from pathlib import Path
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

In [ ]:
# 2. 경로 설정 (본인의 zip 파일 경로로 수정하세요)
zip_path = "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/dacon/no_seg&seg_data.zip"
target_path = "/content/no_seg&seg_data"

# 3. /content 폴더로 복사 (드라이브에서 직접 푸는 것보다 복사 후 푸는 게 빠름)
!cp "{zip_path}" /content/

# 4. 압축 해제 (-q 옵션으로 로그 출력 방지)
zip_name = os.path.basename(zip_path)
!unzip -q "/content/{zip_name}" -d "{target_path}"

print(f"✅ {target_path}에 압축 해제 완료!")

✅ /content/no_seg&seg_data에 압축 해제 완료!


In [ ]:
# ==========================================
# 1. 설정값 (A100 최적화 및 경로)
# ==========================================
CFG = {
    'img_size': 384,
    'batch_size': 16,
    'epochs': 25,
    'backbone_lr': 2e-5,
    'head_lr': 2e-4,
    'n_folds': 5,
    'seed': 42,
    'model_name': 'convnext_base.fb_in22k_ft_in1k_384',

    'orig_root': Path("/content/no_seg&seg_data/no_seg_data"),
    'train_csv': '/content/no_seg&seg_data/no_seg_data/train.csv',
    'dev_csv': '/content/no_seg&seg_data/no_seg_data/dev.csv',
    'test_csv': '/content/no_seg&seg_data/no_seg_data/sample_submission.csv'
}

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [ ]:
# train, dev 합치기 (본인 환경에 맞게 csv 읽는 코드 작성)
train_df = pd.read_csv(CFG['train_csv'])
dev_df = pd.read_csv(CFG['dev_csv'])

train_df['folder'] = 'train'
dev_df['folder'] = 'dev'

all_df = pd.concat([train_df, dev_df], axis=0).reset_index(drop=True)

# 🚨 반드시 unstable을 1로 만들어야 마지막 추론과 꼬이지 않습니다!
all_df['label_int'] = all_df['label'].map({'unstable': 1, 'stable': 0})

# K-Fold 분배용 키
all_df['stratify_key'] = all_df['label'].astype(str) + "_" + all_df['folder']
print(f"데이터 로드 완료: 총 {len(all_df)}개")

데이터 로드 완료: 총 1100개


In [ ]:
train_transform = A.Compose([
    A.Resize(CFG['img_size'], CFG['img_size']),
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1, p=0.8),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, border_mode=0, p=0.6),
    A.GaussianBlur(blur_limit=(3, 7), p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(CFG['img_size'], CFG['img_size']),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

In [ ]:
# 2. 데이터 통합 및 '이중 층화' 키 생성
train_df = pd.read_csv(CFG['train_csv'])
dev_df = pd.read_csv(CFG['dev_csv'])

train_df['folder'] = 'train'
dev_df['folder'] = 'dev'

all_df = pd.concat([train_df, dev_df], axis=0).reset_index(drop=True)
all_df['label_int'] = all_df['label'].map({'unstable': 0, 'stable': 1})

# [핵심] 레이블과 출처를 합친 키 생성 (예: 'stable_train', 'unstable_dev' 등)
all_df['stratify_key'] = all_df['label'].astype(str) + "_" + all_df['folder']

In [ ]:
# 3. Dataset (에러 핸들링 강화)
class StabilityDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        f_path = str(CFG['orig_root'] / row['folder'] / 'front' / f"{row['id']}.png")
        t_path = str(CFG['orig_root'] / row['folder'] / 'top' / f"{row['id']}.png")

        # 순수 RGB 3채널만 로드
        f_img = cv2.cvtColor(cv2.imread(f_path), cv2.COLOR_BGR2RGB)
        t_img = cv2.cvtColor(cv2.imread(t_path), cv2.COLOR_BGR2RGB)

        if self.transform:
            f_img = self.transform(image=f_img)['image']
            t_img = self.transform(image=t_img)['image']

        # Float32로 변경 (BCEWithLogitsLoss 충돌 방지)
        return f_img, t_img, torch.tensor(row['label_int'], dtype=torch.float32)

In [ ]:
# 4. 모델 구조
class UltimateFusionNet(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        # 🌟 in_chans=3 으로 변경 (마스크 제거)
        self.backbone = timm.create_model(model_name, pretrained=True, in_chans=3, num_classes=0)
        feat_dim = self.backbone.num_features

        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feat_dim * 2, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(256, 1),
        )

    def forward(self, front, top):
        f = self.backbone(front)
        t = self.backbone(top)
        return self.head(torch.cat([f, t], dim=1)).squeeze(-1)

In [ ]:
# 5. K-Fold 학습
ACCUM_STEPS = 2  # 배치 16 * 2 = 32 효과
CFG['epochs'] = 40

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])

for fold, (t_idx, v_idx) in enumerate(skf.split(all_df, all_df['stratify_key'])):
    print(f"\n🚀 Fold {fold+1} 시작 (3채널 + WarmRestarts + Accumulation)")
    train_fold, val_fold = all_df.iloc[t_idx], all_df.iloc[v_idx]

    train_ds = StabilityDataset(train_fold, transform=train_transform)
    val_ds = StabilityDataset(val_fold, transform=val_transform)

    train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True, num_workers=4)
    val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=4)

    model = UltimateFusionNet(CFG['model_name']).to(device)
    optimizer = torch.optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': CFG['backbone_lr']},
        {'params': model.head.parameters(), 'lr': CFG['head_lr']},
    ], weight_decay=0.01)

    criterion = nn.BCEWithLogitsLoss()

    # 🌟 1등의 스케줄러 적용
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2
    )
    scaler = GradScaler('cuda')

    best_val_logloss = float('inf')

    for epoch in range(CFG['epochs']):
        model.train()
        train_loss = 0
        optimizer.zero_grad()

        for step, (f, t, l) in enumerate(tqdm(train_loader, desc=f"Fold {fold+1} Ep {epoch+1}")):
            f, t, l = f.to(device), t.to(device), l.to(device)

            with torch.amp.autocast('cuda'):
                logits = model(f, t)
                loss = criterion(logits, l) / ACCUM_STEPS # 🌟 Loss 나누기

            scaler.scale(loss).backward()

            # 🌟 ACCUM_STEPS 주기에 맞춰서 가중치 업데이트
            if (step + 1) % ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            train_loss += loss.item() * ACCUM_STEPS

        # 에포크 단위로 스케줄러 업데이트
        scheduler.step()

        model.eval()
        val_probs, val_labels = [], []
        with torch.no_grad():
            for f, t, l in val_loader:
                f, t = f.to(device), t.to(device)
                logits = model(f, t)
                probs = torch.sigmoid(logits)
                val_probs.extend(probs.cpu().numpy())
                val_labels.extend(l.numpy())

        epoch_logloss = log_loss(val_labels, val_probs, labels=[0, 1])

        print(f"Fold {fold+1} | Ep {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val LogLoss: {epoch_logloss:.6f}")

        if epoch_logloss < best_val_logloss:
            best_val_logloss = epoch_logloss
            torch.save(model.state_dict(), f'best_model_fold{fold+1}.pth')
            print(f"🌟 Fold {fold+1} 최고기록! (LogLoss: {epoch_logloss:.6f})")


🚀 Fold 1 시작 (3채널 + WarmRestarts + Accumulation)


model.safetensors:   0%|          | 0.00/354M [00:00<?, ?B/s]

Fold 1 Ep 1: 100%|██████████| 55/55 [00:25<00:00,  2.15it/s]


Fold 1 | Ep 1 | Train Loss: 0.4066 | Val LogLoss: 0.119910
🌟 Fold 1 최고기록! (LogLoss: 0.119910)


Fold 1 Ep 2: 100%|██████████| 55/55 [00:11<00:00,  4.88it/s]


...

Fold 5 | Ep 38 | Train Loss: 0.0105 | Val LogLoss: 0.003616


Fold 5 Ep 39: 100%|██████████| 55/55 [00:11<00:00,  4.87it/s]


Fold 5 | Ep 39 | Train Loss: 0.0038 | Val LogLoss: 0.001582


Fold 5 Ep 40: 100%|██████████| 55/55 [00:11<00:00,  4.88it/s]


Fold 5 | Ep 40 | Train Loss: 0.0073 | Val LogLoss: 0.000955
🌟 Fold 5 최고기록! (LogLoss: 0.000955)


In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm

# ==========================================
# 1. 추론용 데이터셋 설정
# ==========================================
class StabilityTestDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['id']

        # 1. 마스크(mask) 다 빼고 순수 RGB 원본 이미지만 불러오기
        f_path = str(CFG['orig_root'] / 'test' / 'front' / f"{img_id}.png")
        t_path = str(CFG['orig_root'] / 'test' / 'top' / f"{img_id}.png")

        f_img = cv2.cvtColor(cv2.imread(f_path), cv2.COLOR_BGR2RGB)
        t_img = cv2.cvtColor(cv2.imread(t_path), cv2.COLOR_BGR2RGB)

        # 2. 🌟 핵심 수정 포인트: val_transform을 무조건 거치게 만들기! (정규화 적용)
        if self.transform:
            f_img = self.transform(image=f_img)['image']
            t_img = self.transform(image=t_img)['image']

        return f_img, t_img

# 3. 제출 양식 로드 (경로 주의)
submit = pd.read_csv(CFG['test_csv'])

# 4. 테스트 데이터셋에 반드시 val_transform 넣기
test_ds = StabilityTestDataset(submit, transform=val_transform)
test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=4)

# 5. 5-Fold 모델 로드
models = []
for fold in range(CFG['n_folds']):
    model = UltimateFusionNet(CFG['model_name']).to(device)
    model.load_state_dict(torch.load(f'best_model_fold{fold+1}.pth'))
    model.eval()
    models.append(model)

# 6. TTA(좌우반전) 앙상블 추론 진행
ensemble_probs = []
with torch.no_grad():
    for f, t in tqdm(test_loader, desc="TTA Ensemble Inference"):
        f, t = f.to(device), t.to(device)

        batch_probs = 0
        for model in models:
            # 원본 예측
            logits1 = model(f, t)
            prob1 = torch.sigmoid(logits1)

            # 좌우 반전 이미지 예측 (TTA)
            f_flip = torch.flip(f, dims=[3])
            t_flip = torch.flip(t, dims=[3])
            logits2 = model(f_flip, t_flip)
            prob2 = torch.sigmoid(logits2)

            batch_probs += ((prob1 + prob2) / 2)

        batch_probs /= CFG['n_folds']
        ensemble_probs.extend(batch_probs.cpu().numpy())

ensemble_probs = np.array(ensemble_probs)

# 7. 매운맛 클리핑 (1e-7) 적용
submit['unstable_prob'] = np.clip(ensemble_probs, 1e-7, 1 - 1e-7)
submit['stable_prob'] = 1.0 - submit['unstable_prob']

submit.to_csv('submission_final_SOTA_3ch_WarmRestarts.csv', index=False)
print("🚀 3채널 원복 최종 파일 생성 완료: submission_final_SOTA_3ch_WarmRestarts.csv")

TTA Ensemble Inference: 100%|██████████| 63/63 [02:11<00:00,  2.09s/it]

🚀 3채널 원복 최종 파일 생성 완료: submission_final_SOTA_3ch_WarmRestarts.csv


In [ ]:
import shutil
drive_folder = "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/dacon/가중치/5-fold-v5"

# 폴더가 없으면 생성
if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)
    print(f"📂 새 폴더를 생성했습니다: {drive_folder}")

# 3. 파일 복사 (파일명에 실험 정보 추가)
source_file = "submission_final_SOTA_3ch_WarmRestarts.csv"
destination_file = os.path.join(drive_folder, "submission_final_SOTA_3ch_WarmRestarts.csv")

if os.path.exists(source_file):
    shutil.copy2(source_file, destination_file)
    print(f"✅ 복사 완료: {destination_file}")
else:
    print("❌ 복사할 파일(best_model.pth)을 찾을 수 없습니다.")

✅ 복사 완료: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/dacon/가중치/5-fold-v5/submission_final_SOTA_3ch_WarmRestarts.csv
